# Fine-tune YOLO26n on swimmers (Google Colab), version 2

Run the cells **one by one, top to bottom** (click the ▶ on each, wait until its bracket shows a number before the next). Don't use "Run all": cell 2 waits for your Roboflow key.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

What this does:
1. Gets our repo (scripts + the 294-image eval set).
2. Builds a separate **training** set (TinyPerson + Swimmer detection 2). Images that look like eval images are removed.
3. Tests the normal YOLO26n on the eval set (the "before" number).
4. Fine-tunes YOLO26n gently (~20–30 min).
5. Tests the fine-tuned model on the **same** eval set (the "after" number).
6. Draws precision and recall against the confidence threshold for both models.
7. Downloads the new model, the chart and the results.

**Version 2 changes** (version 1 forgot large, close people: recall fell from 23.8% to 4.9%): people on the beach in TinyPerson are painted over instead of left unboxed, and training keeps the model's original person detector, locks its first 10 layers and uses a lower learning rate.

## 1. Check the GPU and get the code

In [ ]:
!nvidia-smi -L || echo "No GPU! Runtime -> Change runtime type -> T4 GPU"
%cd /content
![ -d /content/MTR-Swimmer-Detection-YOLO26 ] || git clone -q https://github.com/ConanY123/MTR-Swimmer-Detection-YOLO26.git
%cd /content/MTR-Swimmer-Detection-YOLO26
!git pull -q
!pip install -q -U ultralytics roboflow

## 2. Build the training set (about 10–15 minutes)

It asks for your **Roboflow API key** (app.roboflow.com → Settings → API Keys → **Copy** next to the Private API Key; about 20 characters). Click in the box, press Ctrl+V, then Enter. The key is hidden and isn't saved in the notebook.

In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")
!python scripts/build_train_set.py

## 3. "Before": the normal YOLO26n on the eval set

In [ ]:
!python scripts/evaluate_all.py

## 4. Fine-tune (about 20–30 minutes, keep this tab open)

In [ ]:
!python scripts/train_swimmer.py

## 5. "After": the fine-tuned model on the same eval set

In [ ]:
!python scripts/evaluate_all.py --model models/yolo26n_swimmer_v2.pt

## 6. Precision and recall against confidence, both models

In [ ]:
!python scripts/pr_vs_conf.py --models yolo26n.pt models/yolo26n_swimmer_v2.pt
from IPython.display import Image, display
display(Image("dataset/eval_results/pr_vs_conf.png"))

## 7. Download the model and results

In [ ]:
!zip -q -r swimmer_results_v2.zip models/yolo26n_swimmer_v2.pt dataset/eval_results runs/yolo26n_swimmer_v2/results.csv runs/yolo26n_swimmer_v2/results.png
from google.colab import files
files.download("swimmer_results_v2.zip")